# U-Net Radio Deconvolution
**Project:** PSF Deconvolution — Dirty → Clean Image Mapping for Radio Interferometry  
**Author:** Dr. Abubakr Y.A. Ibrahim · ICE-CSIC, Barcelona  
**Repo:** https://github.com/abubakryagob/unet-radio-deconv

---
### How to use this notebook
- **VS Code (local Mac):** open with kernel `radio-unet`, run the ⟳ RECONNECT cells, then continue from your phase.
- **Google Colab (training only):** select T4 GPU, run the ⟳ RECONNECT cells — paths and device adapt automatically.
- **Git rule:** run `nbstripout --install` once (see Phase 1) so outputs are never committed. Only code is tracked.


---
## ⟳ RECONNECTION — Run these at the start of every session

In [ ]:
# ⟳ RECONNECT 1 — Core imports
import os, sys, time, platform
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'OS      : {platform.system()} {platform.machine()}')

In [ ]:
# ⟳ RECONNECT 2 — Device  (Intel Mac=CPU · Colab=CUDA · Apple Silicon=MPS)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

IS_COLAB = 'google.colab' in sys.modules
print(f"Environment : {'Google Colab' if IS_COLAB else 'Local VS Code'}")
print(f'Device      : {device}')
if device.type == 'cpu':
    print('Intel Mac detected — CPU only.')
    print('Phases 1-3 and 5-6 run fine here.')
    print('Phase 4 training: use FAST_DEV=True for logic check, Colab for full run.')

In [ ]:
# ⟳ RECONNECT 3 — Paths  (auto-switches local Mac <-> Colab)
if platform.system() == 'Darwin':                        # local Mac (VS Code)
    PROJECT_ROOT = Path.home() / 'Documents/GitHub/unet-radio-deconv'
else:                                                    # Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/unet-radio-deconv')

DATA_PATH = PROJECT_ROOT / 'data/processed/dataset.h5'
CKPT_DIR  = PROJECT_ROOT / 'models/checkpoints'
FIG_DIR   = PROJECT_ROOT / 'results/figures'

for d in [DATA_PATH.parent, CKPT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data path    : {DATA_PATH}')
print(f'Data exists  : {DATA_PATH.exists()}')

In [ ]:
# ⟳ RECONNECT 4 — Constants and FAST_DEV flag
#
# FAST_DEV = True  -> 128x128, 300 pairs, 3 epochs  (~10 min on Intel CPU)
#                     Use to verify the full pipeline before Colab training.
# FAST_DEV = False -> 256x256, 1500 pairs, 80 epochs (use on Colab GPU)

FAST_DEV = True          # <- change to False on Colab for real training

if FAST_DEV:
    IMG_SIZE               = 128
    N_TRAIN, N_VAL, N_TEST = 200, 50, 50
    N_TOTAL                = 300
    N_EPOCHS               = 3
    BATCH_SIZE             = 8
    BASE_CH                = 32
    print('FAST_DEV=True: 128x128 / 300 pairs / 3 epochs / small U-Net')
else:
    IMG_SIZE               = 256
    N_TRAIN, N_VAL, N_TEST = 1050, 225, 225
    N_TOTAL                = 1500
    N_EPOCHS               = 80
    BATCH_SIZE             = 16
    BASE_CH                = 64
    print('FAST_DEV=False: 256x256 / 1500 pairs / 80 epochs (run on Colab GPU)')

SEED = 42
rng  = np.random.default_rng(SEED)
print(f'Split: Train={N_TRAIN}  Val={N_VAL}  Test={N_TEST}')

In [ ]:
# ⟳ RECONNECT 5 — Verify dataset  (skip if Phase 2 not yet run)
if DATA_PATH.exists():
    with h5py.File(DATA_PATH, 'r') as f:
        n = f['dirty'].shape[0]
        print(f'Dataset OK -- {n} pairs')
        print(f"  dirty : {f['dirty'].shape}  dtype={f['dirty'].dtype}")
        print(f"  File  : {DATA_PATH.stat().st_size/1e6:.1f} MB")
else:
    print(f'Dataset not found at {DATA_PATH}')
    print('-> Run Phase 2 cells below to generate it.')

---
## Phase 1 — Environment Setup
> Run once on first setup. Skip on subsequent sessions.

In [ ]:
# 1-A — Verify packages
import importlib
for pkg in ['torch','torchvision','astropy','skimage','h5py','tqdm','scipy','matplotlib','numpy','pandas']:
    try:
        importlib.import_module(pkg); print(f'  OK  {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg}  <- pip install {pkg}')

In [ ]:
# 1-B — Create project directories (safe to re-run)
for d in ['data/raw','data/processed','models/checkpoints','notebooks','scripts','results/figures']:
    (PROJECT_ROOT/d).mkdir(parents=True, exist_ok=True)
    print(f'  ok  {d}')
print('Folder structure ready.')

In [ ]:
# 1-C — requirements.txt
reqs = (
    'torch>=2.0\ntorchvision>=0.15\nastropy>=5.3\n'
    'scikit-image>=0.21\nh5py>=3.9\ntqdm>=4.66\n'
    'matplotlib>=3.7\nnumpy>=1.24\nscipy>=1.11\n'
    'ipykernel\nnbstripout\n'
)
(PROJECT_ROOT/'requirements.txt').write_text(reqs)
print('requirements.txt written.')

In [ ]:
# 1-D — nbstripout: prevents merge conflicts between Colab and VS Code
#
# Run ONCE in VS Code terminal (Ctrl+`):
#   pip install nbstripout
#   cd ~/Documents/GitHub/unet-radio-deconv
#   nbstripout --install
#
# After this, every git commit automatically strips cell outputs.
# Your local notebook keeps outputs for viewing; only code enters git history.
# This is what prevents conflicts when the same notebook is run in both environments.
print('See comment above -- run nbstripout --install once in the VS Code terminal.')

---
## Phase 2 — Data Pipeline
> Generates synthetic pairs and saves to HDF5.  
> **Skip entirely if RECONNECT 5 reports dataset OK.**

In [ ]:
# 2-A — Extra imports for data generation
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter
print('Data imports OK.')

In [ ]:
# 2-B — Sky model generator
def make_sky_model(size=IMG_SIZE, rng=rng):
    sky = np.zeros((size, size), dtype=np.float32)
    for _ in range(rng.integers(5, 30)):
        x = rng.integers(0, size); y = rng.integers(0, size)
        sky[y, x] += float(rng.exponential(scale=0.3))
    if rng.random() > 0.5:
        for _ in range(rng.integers(1, 4)):
            cx, cy = rng.integers(20, size-20), rng.integers(20, size-20)
            blob = np.zeros((size, size), dtype=np.float32)
            blob[cy, cx] = rng.uniform(0.05, 0.4)
            sky += gaussian_filter(blob, sigma=rng.uniform(5,30)).astype(np.float32)
    if sky.max() > 0: sky /= sky.max()
    return sky

In [ ]:
# 2-C — ALMA-like PSF and dirty image
def make_alma_psf(size=IMG_SIZE, rng=rng):
    uv = np.zeros((size,size), dtype=complex); c = size//2
    si, so = size*rng.uniform(0.05,0.15), size*rng.uniform(0.15,0.40)
    for _ in range(rng.integers(100,500)):
        s = si if rng.random()>0.3 else so
        u = int(np.clip(rng.normal(0,s)+c, 0, size-1))
        v = int(np.clip(rng.normal(0,s)+c, 0, size-1))
        uv[v,u]=1.0; uv[size-v-1,size-u-1]=1.0
    psf = np.abs(np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(uv)))).astype(np.float32)
    if psf.max()>0: psf/=psf.max()
    return psf

def make_dirty_image(sky, psf):
    dirty = fftconvolve(sky, psf, mode='same').astype(np.float32)
    noise = float(np.random.uniform(0.02,0.05)) * dirty.max()
    dirty += np.random.normal(0, noise, dirty.shape).astype(np.float32)
    if dirty.max()>0: dirty/=dirty.max()
    return np.clip(dirty, 0, 1)

In [ ]:
# 2-D — Sanity check: visualise one pair
sky=make_sky_model(); psf=make_alma_psf(); dirty=make_dirty_image(sky,psf)
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,img,title in zip(axes,[sky,psf,dirty],['Clean sky (GT)','PSF','Dirty image']):
    im=ax.imshow(img,cmap='inferno',origin='lower'); ax.set_title(title); ax.axis('off')
    plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
plt.tight_layout(); plt.savefig(FIG_DIR/'sample_pair.png',dpi=150,bbox_inches='tight')
plt.show(); print('Sanity check passed.')

In [ ]:
# 2-E — Generate dataset (guard: skips if file already exists)
# gzip compression: ~750 MB for 1500 pairs (vs 2.5 GB uncompressed)
if DATA_PATH.exists():
    print(f'Dataset already exists at {DATA_PATH} -- skipping.')
    print('Delete the file and re-run to regenerate.')
else:
    print(f'Generating {N_TOTAL} pairs -> {DATA_PATH}')
    with h5py.File(DATA_PATH, 'w') as f:
        kw = dict(compression='gzip', compression_opts=4)
        dd = f.create_dataset('dirty',(N_TOTAL,IMG_SIZE,IMG_SIZE),dtype='float32',**kw)
        cd = f.create_dataset('clean',(N_TOTAL,IMG_SIZE,IMG_SIZE),dtype='float32',**kw)
        for i in tqdm(range(N_TOTAL), desc='Simulating'):
            sky_i=make_sky_model(); dd[i]=make_dirty_image(sky_i,make_alma_psf()); cd[i]=sky_i
    print(f'Saved -- {DATA_PATH.stat().st_size/1e6:.1f} MB')

In [ ]:
# 2-F — PyTorch Dataset class + DataLoaders
from torch.utils.data import Dataset, DataLoader

class RadioImageDataset(Dataset):
    SPLITS = {
        'train':(0, N_TRAIN),
        'val':  (N_TRAIN, N_TRAIN+N_VAL),
        'test': (N_TRAIN+N_VAL, N_TOTAL),
    }
    def __init__(self, h5_path, split='train'):
        self.h5_path=str(h5_path); self.start,self.end=self.SPLITS[split]
        self.length=self.end-self.start; self._file=None
    def _open(self):
        if self._file is None: self._file=h5py.File(self.h5_path,'r')
    def __len__(self): return self.length
    def __getitem__(self, idx):
        self._open(); i=self.start+idx
        return (torch.from_numpy(self._file['dirty'][i]).unsqueeze(0),
                torch.from_numpy(self._file['clean'][i]).unsqueeze(0))

# num_workers=0: avoids HDF5 + multiprocessing fork issues on Mac
dl_train=DataLoader(RadioImageDataset(DATA_PATH,'train'),batch_size=BATCH_SIZE,shuffle=True, num_workers=0)
dl_val  =DataLoader(RadioImageDataset(DATA_PATH,'val'),  batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
dl_test =DataLoader(RadioImageDataset(DATA_PATH,'test'), batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
d,c=next(iter(dl_train))
print(f'DataLoaders ready -- dirty:{d.shape}  clean:{c.shape}')

---
## Phase 3 — U-Net Architecture
> Runs fine on Intel Mac CPU — forward pass is fast.

In [ ]:
# 3-A — Building blocks
class ConvBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True),
            nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True))
    def forward(self,x): return self.block(x)

class DownBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__(); self.net=nn.Sequential(nn.MaxPool2d(2),ConvBlock(in_ch,out_ch))
    def forward(self,x): return self.net(x)

class UpBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.conv=ConvBlock(in_ch,out_ch)
    def forward(self,x,skip):
        x=self.up(x)
        if x.shape!=skip.shape: x=F.interpolate(x,size=skip.shape[2:],mode='bilinear',align_corners=True)
        return self.conv(torch.cat([skip,x],dim=1))

In [ ]:
# 3-B — Full U-Net
class UNet(nn.Module):
    """
    4-level U-Net: dirty radio image (1xHxW) -> clean sky (1xHxW).
    Encoder  : 1 -> b -> 2b -> 4b -> 8b
    Bottleneck: 8b -> 16b
    Decoder  : mirrors encoder with skip connections
    Head     : 1x1 Conv + ReLU  (non-negative sky flux)
    base_ch=32 for FAST_DEV, base_ch=64 for full run.
    """
    def __init__(self,in_ch=1,base_ch=BASE_CH):
        super().__init__(); b=base_ch
        self.enc1=ConvBlock(in_ch,b); self.enc2=DownBlock(b,b*2)
        self.enc3=DownBlock(b*2,b*4); self.enc4=DownBlock(b*4,b*8)
        self.bottleneck=DownBlock(b*8,b*16)
        self.dec4=UpBlock(b*16+b*8,b*8); self.dec3=UpBlock(b*8+b*4,b*4)
        self.dec2=UpBlock(b*4+b*2,b*2);  self.dec1=UpBlock(b*2+b,b)
        self.head=nn.Sequential(nn.Conv2d(b,1,1),nn.ReLU())
    def forward(self,x):
        s1=self.enc1(x); s2=self.enc2(s1); s3=self.enc3(s2); s4=self.enc4(s3)
        x=self.bottleneck(s4)
        x=self.dec4(x,s4); x=self.dec3(x,s3); x=self.dec2(x,s2); x=self.dec1(x,s1)
        return self.head(x)

In [ ]:
# 3-C — Smoke test
model=UNet(in_ch=1,base_ch=BASE_CH).to(device)
dummy=torch.randn(2,1,IMG_SIZE,IMG_SIZE).to(device); out=model(dummy)
n_p=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input : {dummy.shape}'); print(f'Output: {out.shape}')
print(f'Params: {n_p:,}  (~{n_p/1e6:.1f} M)'); print('Architecture OK.')

---
## Phase 4 — Training Pipeline
> **Intel Mac:** `FAST_DEV=True` for a 3-epoch logic check (~10 min on CPU).  
> For the full run: set `FAST_DEV=False`, push to GitHub, open on Colab with T4 GPU.

In [ ]:
# 4-A — Combined MSE + SSIM loss
class SSIMLoss(nn.Module):
    def __init__(self,ws=11,C1=1e-4,C2=9e-4):
        super().__init__(); self.C1,self.C2,self.ws=C1,C2,ws
        g=torch.arange(ws,dtype=torch.float32)-ws//2
        g=torch.exp(-g**2/(2*1.5**2)); g/=g.sum()
        self.register_buffer('kernel',(g[:,None]*g[None,:]).view(1,1,ws,ws))
    def forward(self,pred,target):
        pad=self.ws//2; mu=lambda x:F.conv2d(x,self.kernel,padding=pad)
        mp,mt=mu(pred),mu(target)
        spp=mu(pred**2)-mp**2; stt=mu(target**2)-mt**2; spt=mu(pred*target)-mp*mt
        ssim=((2*mp*mt+self.C1)*(2*spt+self.C2))/((mp**2+mt**2+self.C1)*(spp+stt+self.C2))
        return 1.0-ssim.mean()

class CombinedLoss(nn.Module):
    def __init__(self,lam=0.5):
        super().__init__(); self.mse=nn.MSELoss(); self.ssim=SSIMLoss(); self.lam=lam
    def forward(self,pred,target): return self.mse(pred,target)+self.lam*self.ssim(pred,target)

In [ ]:
# 4-B — Train / validate functions
def train_one_epoch(model,loader,opt,crit,device):
    model.train(); total=0.0
    for d,c in loader:
        d,c=d.to(device),c.to(device); opt.zero_grad()
        loss=crit(model(d),c); loss.backward(); opt.step(); total+=loss.item()
    return total/len(loader)

@torch.no_grad()
def validate(model,loader,crit,device):
    model.eval(); total=0.0
    for d,c in loader:
        d,c=d.to(device),c.to(device); total+=crit(model(d),c).item()
    return total/len(loader)

In [ ]:
# 4-C — Training run
model=UNet(in_ch=1,base_ch=BASE_CH).to(device)
crit=CombinedLoss(lam=0.5).to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-4)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=8,verbose=True)

train_losses,val_losses=[],[]
best_val=float('inf')
print(f'Training on {device} | {N_EPOCHS} epochs | FAST_DEV={FAST_DEV}')

for epoch in range(1,N_EPOCHS+1):
    t0=time.time()
    tl=train_one_epoch(model,dl_train,opt,crit,device)
    vl=validate(model,dl_val,crit,device)
    sched.step(vl); train_losses.append(tl); val_losses.append(vl)
    print(f'Epoch {epoch:3d}/{N_EPOCHS}  train {tl:.5f}  val {vl:.5f}  ({time.time()-t0:.1f}s)')
    if vl<best_val:
        best_val=vl
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),'val_loss':vl},
                   CKPT_DIR/'best_model.pth')
        print(f'  checkpoint saved (val={vl:.5f})')
    if epoch%10==0:
        torch.save(model.state_dict(),CKPT_DIR/f'model_epoch_{epoch:03d}.pth')

In [ ]:
# 4-D — Loss curves
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(train_losses,label='Train',linewidth=1.5); ax.plot(val_losses,label='Val',linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE + SSIM loss')
ax.set_title('Training curve'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR/'loss_curve.png',dpi=150); plt.show()
print(f'Best val loss: {best_val:.5f}')

---
## Phase 5 — Evaluation & Comparison with CLEAN
> Runs fine on Intel Mac CPU — no training loop.

In [ ]:
# 5-A — Load best checkpoint
ckpt=torch.load(CKPT_DIR/'best_model.pth',map_location=device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print(f"Loaded epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.5f}")

In [ ]:
# 5-B — Hogbom CLEAN baseline
def hogbom_clean(dirty_img,psf,gain=0.1,n_iter=500,threshold=0.01):
    res=dirty_img.copy().astype(np.float64); comp=np.zeros_like(res)
    cy,cx=psf.shape[0]//2,psf.shape[1]//2; stop=threshold*np.abs(res).max()
    for _ in range(n_iter):
        pv=res.max()
        if pv<stop: break
        py,px=np.unravel_index(res.argmax(),res.shape)
        psf_s=np.roll(np.roll(psf,py-cy,axis=0),px-cx,axis=1)
        res-=gain*pv*psf_s; comp[py,px]+=gain*pv
    cb=np.zeros_like(psf); cb[cy,cx]=1.0
    out=fftconvolve(comp,gaussian_filter(cb,sigma=2.0),mode='same')+res
    out=np.clip(out,0,None).astype(np.float32)
    if out.max()>0: out/=out.max()
    return out

In [ ]:
# 5-C — Metrics on the test set
from skimage.metrics import peak_signal_noise_ratio as psnr_fn, structural_similarity as ssim_fn

res_dict={'unet':{'ssim':[],'psnr':[]},'clean':{'ssim':[],'psnr':[]}}
with torch.no_grad():
    for db,cb in tqdm(dl_test,desc='Evaluating'):
        pb=model(db.to(device)).cpu().numpy()
        cn,dn=cb.numpy(),db.numpy()
        for b in range(len(cn)):
            gt=cn[b,0]; pred=np.clip(pb[b,0],0,1); dirt=dn[b,0]
            cl=hogbom_clean(dirt,make_alma_psf(size=dirt.shape[-1]),n_iter=300)
            res_dict['unet']['ssim'].append(ssim_fn(gt,pred,data_range=1.0))
            res_dict['unet']['psnr'].append(psnr_fn(gt,pred,data_range=1.0))
            res_dict['clean']['ssim'].append(ssim_fn(gt,cl,data_range=1.0))
            res_dict['clean']['psnr'].append(psnr_fn(gt,cl,data_range=1.0))
print('\n-- Test results --')
for m in ('unet','clean'):
    print(f"  {m.upper():5s}  SSIM {np.mean(res_dict[m]['ssim']):.4f}  PSNR {np.mean(res_dict[m]['psnr']):.2f} dB")

In [ ]:
# 5-D — Side-by-side: dirty | CLEAN | U-Net | GT
dv,cv=next(iter(dl_test))
with torch.no_grad(): pv=model(dv.to(device)).cpu()
n_show=4; fig,axes=plt.subplots(n_show,4,figsize=(13,n_show*3))
for i in range(n_show):
    gt=cv[i,0].numpy(); dirt=dv[i,0].numpy(); pred=pv[i,0].numpy()
    cl=hogbom_clean(dirt,make_alma_psf(size=dirt.shape[-1]),n_iter=200)
    for j,(img,title) in enumerate(zip([dirt,cl,pred,gt],['Dirty','CLEAN','U-Net','GT'])):
        ax=axes[i,j]; ax.imshow(img,cmap='inferno',origin='lower',vmin=0,vmax=1)
        if i==0: ax.set_title(title,fontsize=11,fontweight='bold')
        ax.axis('off')
plt.suptitle('Dirty | CLEAN | U-Net | Ground truth',fontsize=12,y=1.01)
plt.tight_layout(); plt.savefig(FIG_DIR/'qualitative_comparison.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
# 5-E — Residual maps
fig,axes=plt.subplots(2,4,figsize=(13,6))
for i in range(4):
    gt=cv[i,0].numpy(); pred=pv[i,0].numpy(); dirt=dv[i,0].numpy()
    cl=hogbom_clean(dirt,make_alma_psf(size=dirt.shape[-1]),n_iter=200)
    vmax=max(np.abs(gt-cl).max(),np.abs(gt-pred).max())
    axes[0,i].imshow(np.abs(gt-cl),cmap='viridis',origin='lower',vmin=0,vmax=vmax)
    axes[0,i].set_title(f'|CLEAN-GT| #{i+1}',fontsize=9); axes[0,i].axis('off')
    axes[1,i].imshow(np.abs(gt-pred),cmap='viridis',origin='lower',vmin=0,vmax=vmax)
    axes[1,i].set_title(f'|U-Net-GT| #{i+1}',fontsize=9); axes[1,i].axis('off')
plt.suptitle('Residuals: row 1=CLEAN  row 2=U-Net  (darker=better)',fontsize=11)
plt.tight_layout(); plt.savefig(FIG_DIR/'residual_maps.png',dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
# 5-F — Metrics summary table
import pandas as pd
df=pd.DataFrame({
    'Method':['U-Net','Hogbom CLEAN'],
    'SSIM mean':[f"{np.mean(res_dict['unet']['ssim']):.4f}",f"{np.mean(res_dict['clean']['ssim']):.4f}"],
    'SSIM std': [f"{np.std(res_dict['unet']['ssim']):.4f}", f"{np.std(res_dict['clean']['ssim']):.4f}"],
    'PSNR dB':  [f"{np.mean(res_dict['unet']['psnr']):.2f}", f"{np.mean(res_dict['clean']['psnr']):.2f}"],
})
print(df.to_string(index=False))
df.to_csv(PROJECT_ROOT/'results/metrics_summary.csv',index=False)
print('Saved -> results/metrics_summary.csv')

---
## Phase 6 — GitHub Deployment
> Commit via VS Code Source Control panel (branch icon, left sidebar).

In [ ]:
# 6-A — .gitignore
gi = '# Data -- stored locally, never committed\ndata/\n*.h5\n*.hdf5\n\n'
gi += '# Model weights -- attach to GitHub Releases instead\nmodels/checkpoints/*.pth\n\n'
gi += '# Python\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n\n# macOS\n.DS_Store\n'
(PROJECT_ROOT/'.gitignore').write_text(gi)
print('.gitignore written.')

In [ ]:
# 6-B — Commit via VS Code Source Control panel:
#
#   1. Click the branch icon in the left sidebar
#   2. Click + next to each file to stage it
#   3. Message: 'feat: complete U-Net pipeline all 6 phases'
#   4. Click Commit -> Sync
#
# OR in VS Code terminal (Ctrl+`):
#   git add notebooks/ scripts/ results/figures/ README.md .gitignore requirements.txt
#   git commit -m 'feat: complete U-Net pipeline all 6 phases'
#   git tag -a v0.1 -m 'First working model'
#   git push origin main --tags
print('See comment above.')